In [ ]:
import pandas as pd
import os
import numpy as np

def reverse_hypotheses_and_label(input_csv_path, output_csv_path):
    """
    Reads a CSV file, swaps 'hypothesis_1' and 'hypothesis_2',
    reverses the 'label' accordingly (ensuring integer output for processed numeric labels),
    and saves the result.

    Args:
        input_csv_path (str): Path to the input CSV file.
        output_csv_path (str): Path to save the modified CSV file.
    """
    try:
        # Read the CSV file
        df = pd.read_csv(input_csv_path)
        print(f"Successfully loaded '{input_csv_path}'.")
        print("\nOriginal DataFrame head:")
        print(df.head())
        print(f"Original label column dtype: {df['label'].dtype}")


        # Verify necessary columns exist
        required_columns = ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing required columns: {', '.join(missing_columns)}")
            return

        # 1. Swap hypothesis_1 and hypothesis_2
        df[['hypothesis_1', 'hypothesis_2']] = df[['hypothesis_2', 'hypothesis_1']]
        print("\nDataFrame head after swapping hypotheses:")
        print(df.head())

        # 2. Reverse the label "accordingly"
        original_label_dtype = df['label'].dtype
        transformed_numeric_label = False # Flag to track if numeric labels were processed

        if pd.api.types.is_numeric_dtype(df['label']):
            unique_labels = df['label'].unique()
            # Convert to a set of integers if possible for comparison, handling potential floats like 1.0
            try:
                unique_labels_as_int_set = set(map(int, unique_labels))
            except ValueError:
                unique_labels_as_int_set = set(unique_labels)


            if unique_labels_as_int_set == {0, 1}:
                print("\nLabel column appears to be binary numeric (0/1 or 0.0/1.0). Flipping values.")
                df['label'] = 1 - df['label']
                transformed_numeric_label = True
            elif unique_labels_as_int_set == {1, 2}:
                print("\nLabel column contains numeric values 1 and 2 (or 1.0 and 2.0). Swapping values.")
                label_map = {1: 2, 2: 1, 1.0: 2, 2.0: 1} # Ensures integer output
                df['label'] = df['label'].map(label_map)
                transformed_numeric_label = True
            else:
                print(f"\nLabel column is numeric (type: {original_label_dtype}) but not 0/1 or 1/2. "
                      "The script currently only has specific reversal logic for 0/1 and 1/2 numeric labels. "
                      "Labels will remain unchanged. Please review if this is appropriate or modify the script.")

            if transformed_numeric_label:
                if df['label'].isnull().any(): # Check for NaNs introduced by map if a label wasn't in keys
                    print("\nWarning: Some labels became NaN after transformation. Cannot convert to int. Check mapping logic and source data.")
                else:
                    print("Ensuring processed numeric labels are integers.")
                    df['label'] = df['label'].astype(int)

        elif pd.api.types.is_bool_dtype(df['label']):
            print("\nLabel column is boolean. Negating values (True->False, False->True).")
            df['label'] = ~df['label']

        elif pd.api.types.is_string_dtype(df['label']):
            print(f"\nLabel column is string (type: {original_label_dtype}). "
                  "Reversing string labels requires a specific mapping.")
            # Example:
            # label_reversal_map = {"entailment": "contradiction", "contradiction": "entailment", "neutral": "neutral"}
            # df['label'] = df['label'].map(label_reversal_map).fillna(df['label'])
            print("String labels are currently unchanged. You need to implement custom mapping logic if required.")

        else:
            print(f"\nLabel column type ({original_label_dtype}) is not explicitly handled for reversal. "
                  "The label column remains unchanged.")

        print("\nDataFrame head after reversing label (if applicable):")
        print(df.head())
        print(f"Final label column dtype: {df['label'].dtype}")


        # Save the modified DataFrame
        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
        df.to_csv(output_csv_path, index=False)
        print(f"\nSuccessfully processed the file and saved the output to '{output_csv_path}'.")

    except FileNotFoundError:
        print(f"Error: The file '{input_csv_path}' was not found. Please ensure the path and filename are correct.")
    except pd.errors.EmptyDataError:
        print(f"Error: The file '{input_csv_path}' is empty.")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == '__main__':
    # --- Configuration for your CSV file ---
    data_folder = 'data'  # Assumes a 'data' folder in your Colab environment or current directory

    # **IMPORTANT**: Specify your input and output file names here
    your_input_filename = 'validation_data_arabic.csv'  # <--- CHANGE THIS TO YOUR ACTUAL INPUT FILENAME
    your_output_filename = 'your_data_reversed.csv' # <--- CHANGE THIS TO YOUR DESIRED OUTPUT FILENAME

    input_csv_path = os.path.join(data_folder, your_input_filename)
    output_csv_path = os.path.join(data_folder, your_output_filename)

    # --- Process your CSV file ---
    print(f"--- Processing your specified file: {your_input_filename} ---")

    # Ensure the 'data' folder exists if the output path includes it
    # (os.makedirs in the function will create it if it doesn't exist,
    # but good practice to ensure base 'data' folder for input if needed)
    if not os.path.exists(data_folder) and data_folder: # Check if data_folder is not empty string
        try:
            os.makedirs(data_folder)
            print(f"Created directory: {data_folder} (if it wasn't there for input file placement)")
        except OSError as e:
            print(f"Could not create directory {data_folder}: {e}. Please ensure it exists or adjust path.")
            # Optionally exit if directory creation is critical and fails
            # import sys
            # sys.exit(1)


    print(f"Attempting to process input file: {input_csv_path}")
    print(f"Output will be saved to: {output_csv_path}")

    reverse_hypotheses_and_label(input_csv_path, output_csv_path)

    print("\n--- Script finished ---")
    print(f"Instructions:")
    print(f"1. Place your CSV file (e.g., '{your_input_filename}') into the '{data_folder}' folder (or adjust path).")
    print(f"2. Ensure 'your_input_filename' and 'your_output_filename' variables in the script are correctly set.")
    print(f"3. Run the script.")


--- Processing your specified file: validation_data_arabic.csv ---
Attempting to process input file: data/validation_data_arabic.csv
Output will be saved to: data/your_data_reversed.csv
Successfully loaded 'data/validation_data_arabic.csv'.

Original DataFrame head:
                                       observation_1  \
0  بدأ رون وظيفته الجديدة كمُهَيء المناظر الطبيعي...   
1                             عاشت ساندي في نيويورك.   
2  جاءت والدة مريم إلى المنزل بمزيد من الموز مما ...   
3                            كان جيم يعمل على مشروع.   
4                          كان شان جالسًا على مكتبه.   

                                   observation_2  \
0         تم فصْل رون على الفور بسبب عدم الخضوع.   
1                             كانت ساندي مستعدة.   
2  كانت تلك أفضل طريقة على الإطلاق لتناول الموز!   
3                  لحسن الحظ، وجدها على رف قريب.   
4         بعد دقيقة، تمكن من إعادة تجميع الكرسي.   

                                        hypothesis_1  \
0            رون يتجاهل أوا